<a href="https://colab.research.google.com/github/minji25-hue/NVIDIA/blob/20250404/yolo_v8_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

yolo v8로 교통량 분석해보기.

In [26]:
# 1️⃣ 필수 라이브러리 설치
!apt-get install -y libgl1-mesa-glx
!pip install -U yt-dlp ultralytics opencv-python

import yt_dlp
import cv2
import time
import os
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# 2️⃣ YouTube 영상 다운로드 (yt-dlp 사용)
def download_youtube_video(url, output_path="traffic.mp4"):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best',
        'outtmpl': output_path,  # 저장할 파일 경로
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print(f"✅ YouTube 영상 다운로드 완료: {output_path}")
    except Exception as e:
        print(f"❌ YouTube 다운로드 오류: {e}")

# 🎯 🔄 **새로운 YouTube 영상 URL 적용**
video_url = 'https://www.youtube.com/watch?v=QedNLyleW1w'  # 새로운 영상 URL
download_youtube_video(video_url, "/content/traffic.mp4")

# 3️⃣ YOLOv8x 모델 로드
model = YOLO('yolov8x.pt')

# 4️⃣ 차량 감지 및 카운팅
cap = cv2.VideoCapture("/content/traffic.mp4")

if not cap.isOpened():
    print("❌ 비디오 파일을 열 수 없습니다. 다운로드가 실패했을 가능성이 있습니다.")
else:
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)  # FPS 정보 가져오기
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # 총 프레임 수
    duration = total_frames / fps  # 전체 영상 길이(초)

    print(f"🎥 영상 정보: {duration:.2f}초, {fps:.2f} FPS, 총 {total_frames} 프레임")

    # ✅ MP4V 코덱 사용
    out = cv2.VideoWriter('/content/output.mp4', cv2.VideoWriter_fourcc(*'MP4V'), int(fps), (frame_width, frame_height))

    # VideoWriter 정상 작동 여부 확인
    if not out.isOpened():
        print("❌ VideoWriter가 정상적으로 열리지 않았습니다. 코덱을 확인하세요.")
    else:
        print("✅ VideoWriter 초기화 성공")

    car_ids = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 🚗 차량 감지 및 추적
        results = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=[2,3,5,7], conf=0.5)

        if results and results[0].boxes.id is not None:
            for box_id in results[0].boxes.id.cpu().numpy().astype(int):
                car_ids.add(box_id)

        # 감지된 프레임 저장
        annotated_frame = results[0].plot() if results else frame
        out.write(annotated_frame)

    cap.release()
    out.release()

    # 5️⃣ 결과 출력
    print(f"🔹 전체 영상에서 통과한 차량 수: {len(car_ids)}대")
    print("\n📌 결과 영상 확인 방법:")
    print("1. 좌측 폴더 아이콘 📁 클릭")
    print("2. `/content/output.mp4` 파일 우클릭 → [다운로드] 또는 [미리보기]")

    # 6️⃣ output.mp4 파일 존재 여부 확인 후 다운로드
    output_path = "/content/output.mp4"
    if os.path.exists(output_path):
        print("✅ output.mp4 파일이 정상적으로 저장되었습니다.")
        from google.colab import files
        files.download(output_path)
    else:
        print("❌ output.mp4 파일이 생성되지 않았습니다. OpenCV 설정을 확인하세요.")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
[youtube] Extracting URL: https://www.youtube.com/watch?v=QedNLyleW1w
[youtube] QedNLyleW1w: Downloading webpage
[youtube] QedNLyleW1w: Downloading tv client config
[youtube] QedNLyleW1w: Downloading player 73381ccc-main
[youtube] QedNLyleW1w: Downloading tv player API JSON
[youtube] QedNLyleW1w: Downloading ios player API JSON
[youtube] QedNLyleW1w: Downloading m3u8 information
[info] QedNLyleW1w: Downloading 1 format(s): 137+140
[download] Destination: /content/traffic.f137.mp4
[download] 100% of   19.30MiB in 00:00:00 at 31.40MiB/s  
[download] Destination: /content/traffic.f140.m4a
[download] 100% of  617.92KiB in 00:00:00 at 16.51MiB/s  
[Merger] Merging formats into "/content/traffic.mp4"
Deleting original file /content/traffic.f137.mp4 (pass -k to k

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>